In [1]:
import openai
from qdrant_client import QdrantClient

### Embedding function

In [9]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval Function

In [11]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [10]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [13]:
retrieved_context = retrieve_data("Do you have a USB connectable fan for hot summers.", k=10)

In [14]:
retrieved_context

{'retrieved_context_ids': ['B0BRJS644Z',
  'B0BXC72RLD',
  'B099N9F3FP',
  'B0C8DBH7ZT',
  'B0BM9THPDQ',
  'B0CF57H28T',
  'B0C9QZS95R',
  'B0BYD7PGV1',
  'B09VDLH5M6',
  'B0CC4HBS85'],
 'retrieved_context': ['Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that supp

### Format retrieved context function

In [15]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [16]:
preprocessed_context = process_context(retrieved_context)

In [17]:
print(preprocessed_context)

- ID: B0BRJS644Z, rating: 4.7, description: Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】 This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed Control Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】 Rubber Feet on the fan raise it up enough off of the surface it is sitti

### Create prompt template function

In [18]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [19]:
prompt = build_prompt(preprocessed_context, "Do you have a USB connectable fan for hot summer?")

In [20]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- ID: B0BRJS644Z, rating: 4.7, description: Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】 This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed Control Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility U

### Generate answer function

In [21]:
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content

In [22]:
print(generate_answer(prompt))

Yes. We have USB-connectable fans that work well for hot summer:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (ID: B0BRJS644Z, rating 4.7)
- 120mm fan for cooling electronics (router/TV box/modem/DVR, etc.)
- USB-powered via 3.3 ft USB cable
- Speed controller on the cord: off / low / medium / high
- Includes dust filters and rubber feet

2) HZD Desk Fan Rechargeable, Mini Portable Fan (ID: B0BXC72RLD, rating 4.3)
- Mini personal fan for desks and travel
- 3 speeds (low/medium/high)
- USB-powered via 4.9 ft USB cable (note: it does not come with a battery)
- Quiet operation (noise listed as less than 50 dB)
- Includes LED light

Which do you need: a larger 120mm fan for device cooling, or a small portable desk fan for personal use?


### Combined RAG pipeline

In [23]:
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [24]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

Yes. We have a few USB-connectable fans that work well for hot summers:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (ID: B0BRJS644Z)
- 120mm fan for cooling electronics
- Speed control on the cord (off/low/medium/high)
- Powered via USB (3.3 ft cable)
- Includes rubber feet and a dust filter

2) HZD Mini Portable Rechargeable Fan (ID: B0BXC72RLD)
- Compact personal fan (USB-powered)
- 3 speeds (low/medium/high)
- Noise less than 50d
- Comes with a USB cable (4.9 ft) and is compatible with common USB power sources

If you tell me whether you want a small personal desk fan or a larger 120mm airflow fan for a device (TV box/router/laptop area), I can recommend the best match.


In [25]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Sure—here are earphones with a rating above 4 that are available:

1) B0CH8DRD6K (rating 4.9)
2 Pack iPhone Wired Headphones with 3.5mm Jack (Apple MFi Certified), with microphone and volume control.

2) B0BRV544MV (rating 4.4)
Wekily Bluetooth 5.3 Wireless Earbuds, 40H playtime (with charging case), deep bass, IPX7 waterproof, 4-mic clear call, LED display.

3) B0BBF2VC6X (rating 4.2)
Volkano Ninja Kids Earphones (wired) with 3.5mm jack, safe volume limit 85dB, includes a carry case and keyring.

4) B09KQP2H7N (rating 4.2)
Kids Wireless/ Wired Headphones (Bluetooth 5.0 + 3.5mm jack), adjustable headband, built-in microphone, up to 7 hours (battery listed).

If you tell me whether you want wired or wireless (and for iPhone or Android), I can narrow it down to the best match.


In [26]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))

Sure—here are earphones/headphones from the available products that have a rating below 4.0:

1) pamu S29 Wireless Earbuds (Bluetooth 5.2, ANC/ENC, USB-C fast charge) — rating 3.9  
2) RUNAR RNR1 Wireless Neckband Running Headphones (Bluetooth 5.0, sweatproof, TF/SD compatible) — rating 3.8  

If you tell me what you need most (sports/waterproof, best calls, active noise cancellation, or wired vs wireless), I can help you pick between these two.
